In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install scikit-bio

In [3]:
import json
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D

from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist, squareform

from skbio.stats.distance import DistanceMatrix, permanova

from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering, KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

from sklearn.ensemble import IsolationForest

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import roc_curve, roc_auc_score, silhouette_score, r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [4]:
df = pd.read_excel("train_data.xlsx")

In [5]:
df

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
0,20,1.20,1.0,100,80,100,70,1
1,23,1.10,0.9,98,78,98,78,1
2,22,1.40,1.1,96,85,99,79,1
3,27,1.20,1.2,90,89,90,83,1
4,21,1.80,0.8,92,95,96,82,1
...,...,...,...,...,...,...,...,...
79,85,0.32,2.5,75,47,44,31,0
80,77,0.35,2.2,61,41,32,33,0
81,74,0.27,2.1,67,42,32,32,0
82,67,0.60,2.3,66,46,37,30,0


In [6]:
# Shuffle original dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [7]:
# Standardization
def standardization(x):
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)
    return x_scaled, scaler

In [11]:
def isolation_forest(df, label_col=None, contamination=0.1):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # Remove label if present
    if label_col is not None:
        feature_cols = [col for col in feature_cols if col != label_col]

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for Isolation Forest.")

    # missing value check
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    # feature matrix
    X = df[feature_cols]

    # standardization (your existing function)
    X_scaled, scaler = standardization(X)

    # Isolation Forest model
    model = IsolationForest(contamination=contamination, random_state=42)

    model.fit(X_scaled)
    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/RepoGo_Rasp/models/isolation_forest_model.pkl"

    joblib.dump(artifact, model_path)

    print("Model saved successfully at:", model_path)
    print(feature_cols)
    return

In [12]:
isolation_forest(df, label_col="Group", contamination=0.05)

Model saved successfully at: /content/drive/MyDrive/RepoGo_Rasp/models/isolation_forest_model.pkl
['Permeability', 'LDL_uptake', 'Total_ROS', 'Vascular_Marker', 'Cell_Signalling', 'Tube_formation', 'In_vivo_recovery']


# Predictive Modelling

In [ ]:
# Validation
def validate_input(df, label_col):
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")
    if df.empty:
        raise ValueError("Dataset is empty.")
    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataset.")
    if df[label_col].nunique() != 2:
        raise ValueError("Binary classification requires exactly 2 classes.")
    if len(df) < 10:
        raise ValueError("Dataset too small for modeling.")

# 1). Simple Logistic Regression

In [ ]:
def train_logistic(df, label_col, test_data_size=0.28, model_path="/content/drive/MyDrive/RepoGo_Rasp/models/logistic_model.pkl"):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_data_size,
        random_state=42,
        stratify=y
    )

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Cross Validation
    n_samples = len(X_train)
    num_splits = 3 if n_samples < 50 else 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    model = LogisticRegression(
        penalty=None,
        solver="newton-cg",
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )

    cv_scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=skf,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    # Train Final Model
    model.fit(X_train_scaled, y_train)

    # Predictions (default threshold = 0.5)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Save Pipeline
    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    joblib.dump(artifact, model_path)

    # Final Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "cv_auc_mean": cv_mean,
            "cv_auc_std": cv_std
        },
        "model_results": {
            "model_type": "Plain Logistic Regression",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        },
        "saved_model_path": model_path
    }

In [ ]:
train_logistic(df, label_col="Group", test_data_size=0.28, model_path="/content/drive/MyDrive/RepoGo_Rasp/models/logistic_model.pkl")

{'data_summary': {'num_samples': 84,
  'num_features': 7,
  'class_distribution': {0: 42, 1: 42}},
 'model_selection': {'cv_folds': 5, 'cv_auc_mean': 1.0, 'cv_auc_std': 0.0},
 'model_results': {'model_type': 'Plain Logistic Regression',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[12, 0], [0, 12]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 12.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 12.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 24.0},
   'weighted avg': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 24.0}},
  'roc_curve': {'fpr': [0.0, 0.0, 0.0, 1.0],
   'tpr': [0.0, 0.08333333333333333, 1.0, 1.0]},
  'coefficients': {'In_vivo_recovery': 2.173396436703604,
   'Tube_formation': 2.130299165455939,
   'Cell_Signalling': 1.7490441261631313,
   'Vascular_Marker': 1.130243730352579

In [ ]:
def retrain_on_full_data(df, label_col, model_path):
    validate_input(df, label_col)

    artifact = joblib.load(model_path)

    old_model = artifact["model"]
    feature_cols = artifact["features"]

    # Feature check
    missing_cols = set(feature_cols) - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing features: {missing_cols}")

    X = df[feature_cols].copy()
    y = df[label_col]

    # New scaler (clean)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Fresh model (clean training)
    model = LogisticRegression(
        penalty=old_model.penalty,
        solver=old_model.solver,
        max_iter=old_model.max_iter,
        class_weight=old_model.class_weight,
        random_state=42
    )

    # Train on full data
    model.fit(X_scaled, y)

    # Save final model
    updated_artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    joblib.dump(updated_artifact, model_path)

    return {
        "message": "Model retrained on full dataset",
        "num_samples": len(df)
    }

In [ ]:
retrain_on_full_data(df, "Group", "/content/drive/MyDrive/RepoGo_Rasp/models/logistic_model.pkl")

{'message': 'Model retrained on full dataset', 'num_samples': 84}

## 2). Ridge

In [ ]:
def tune_ridge_logistic_classification(df, label_col, test_data_size=0.28):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Adaptive CV Folds
    n_samples = len(X_train)

    if n_samples < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)
    if total_samples < 50:
        # Conservative: prevents extreme over/underfitting
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        # Wide search for big data
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = {"C": c_range}

    base_model = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_scaled, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    c_values = cv_results["param_C"].astype(float)
    train_scores = cv_results["mean_train_score"]
    val_scores = cv_results["mean_test_score"]

    # Sort by C for frontend plotting
    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(best_model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Ridge)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        }
    }

In [ ]:
tune_ridge_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


{'data_summary': {'num_samples': 84,
  'num_features': 7,
  'class_distribution': {0: 42, 1: 42}},
 'model_selection': {'cv_folds': 5,
  'best_C': 0.01,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.01,
    0.019306977288832496,
    0.0372759372031494,
    0.07196856730011521,
    0.13894954943731375,
    0.2682695795279725,
    0.517947467923121,
    1.0,
    1.9306977288832496,
    3.727593720314938,
    7.196856730011514,
    13.894954943731374,
    26.826957952797247,
    51.794746792312075,
    100.0],
   'train_auc': [1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0],
   'validation_auc': [1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0]}},
 'model_results': {'model_type': 'Logistic Regression (Ridge)',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[12, 0], [0, 12]],
  'classification_repo

In [ ]:
# Hyperparameter tuning
tuning_results = tune_ridge_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

best_C = tuning_results["model_selection"]["best_C"]

Fitting 5 folds for each of 15 candidates, totalling 75 fits


In [ ]:
best_C

0.01

In [ ]:
def retrain_ridge_logistic(df, label_col, best_C):

    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=best_C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_scaled, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/RepoGo_Rasp/models/ridge_logistic_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_ridge_logistic(df, "Group", best_C)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/RepoGo_Rasp/models/ridge_logistic_model.pkl',
 'num_samples': 84,
 'num_features': 7}

## 3). Lasso

In [ ]:
def tune_lasso_logistic_classification(df, label_col, test_data_size=0.28):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Adaptive CV Folds
    n_samples = len(X_train)

    if n_samples < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)
    if total_samples < 50:
        # Conservative: prevents extreme over/underfitting
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        # Wide search for big data
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = {"C": c_range}

    base_model = LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_scaled, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    c_values = cv_results["param_C"].astype(float)
    train_scores = cv_results["mean_train_score"]
    val_scores = cv_results["mean_test_score"]

    # Sort by C for frontend plotting
    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(best_model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Ridge)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        }
    }

In [ ]:
tune_lasso_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


{'data_summary': {'num_samples': 84,
  'num_features': 7,
  'class_distribution': {0: 42, 1: 42}},
 'model_selection': {'cv_folds': 5,
  'best_C': 0.07196856730011521,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.01,
    0.019306977288832496,
    0.0372759372031494,
    0.07196856730011521,
    0.13894954943731375,
    0.2682695795279725,
    0.517947467923121,
    1.0,
    1.9306977288832496,
    3.727593720314938,
    7.196856730011514,
    13.894954943731374,
    26.826957952797247,
    51.794746792312075,
    100.0],
   'train_auc': [0.5,
    0.5,
    0.5,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0],
   'validation_auc': [0.5,
    0.5,
    0.5,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0]}},
 'model_results': {'model_type': 'Logistic Regression (Ridge)',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[12, 0], [0, 12]],
  'clas

In [ ]:
# Hyperparameter tuning
tuning_results = tune_lasso_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

best_C = tuning_results["model_selection"]["best_C"]

Fitting 5 folds for each of 15 candidates, totalling 75 fits


In [ ]:
best_C

0.07196856730011521

In [ ]:
def retrain_lasso_logistic(df, label_col, best_C):

    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=best_C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_scaled, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/RepoGo_Rasp/models/lasso_logistic_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_lasso_logistic(df, "Group", best_C)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/RepoGo_Rasp/models/lasso_logistic_model.pkl',
 'num_samples': 84,
 'num_features': 7}

## 4). ElasticNet

In [ ]:
def tune_elasticnet_logistic_classification(df, label_col, test_data_size=0.28):

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    if len(X_train) < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)

    if total_samples < 50:
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        c_range = np.logspace(-3, 3, 20)

    # Elastic Net requires l1_ratio
    l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

    parameter_grid = {
        "C": c_range,
        "l1_ratio": l1_ratios
    }

    # Base Model
    base_model = LogisticRegression(penalty="elasticnet", solver="saga", max_iter=2000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_scaled, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    # For frontend, show curve for best l1_ratio
    best_l1_ratio = grid.best_params_["l1_ratio"]

    filtered_results = cv_results[cv_results["param_l1_ratio"] == best_l1_ratio]

    c_values = filtered_results["param_C"].astype(float)
    train_scores = filtered_results["mean_train_score"]
    val_scores = filtered_results["mean_test_score"]

    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(best_model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_l1_ratio": float(best_l1_ratio),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Elastic Net)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        }
    }

In [ ]:
# Hyperparameter tuning
tuning_results = tune_elasticnet_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

best_C = tuning_results["model_selection"]["best_C"]
best_l1_ratio = tuning_results["model_selection"]["best_l1_ratio"]

Fitting 5 folds for each of 75 candidates, totalling 375 fits


In [ ]:
best_C

0.01

In [ ]:
best_l1_ratio

0.1

In [ ]:
def retrain_elasticnet_logistic(df, label_col, best_C, best_l1_ratio):

    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=best_C,
        l1_ratio=best_l1_ratio,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_scaled, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/RepoGo_Rasp/models/elasticnet_logistic_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_elasticnet_logistic(df, "Group", best_C, best_l1_ratio)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/RepoGo_Rasp/models/elasticnet_logistic_model.pkl',
 'num_samples': 84,
 'num_features': 7}

In [ ]:
def retrain_logistic_wrapper(df, label_col, model_type="simple", model_path=None, best_C=None, best_l1_ratio=None):
    """
    Wrapper function to call appropriate retraining function
    """

    if model_type == "simple":
        if model_path is None:
            raise ValueError("model_path is required for simple logistic retraining")
        return retrain_on_full_data(df, label_col, model_path)

    elif model_type == "ridge":
        if best_C is None:
            raise ValueError("best_C is required for ridge logistic")
        return retrain_ridge_logistic(df, label_col, best_C)

    elif model_type == "lasso":
        if best_C is None:
            raise ValueError("best_C is required for lasso logistic")
        return retrain_lasso_logistic(df, label_col, best_C)

    elif model_type == "elasticnet":
        if best_C is None or best_l1_ratio is None:
            raise ValueError("best_C and best_l1_ratio are required for elasticnet logistic")
        return retrain_elasticnet_logistic(df, label_col, best_C, best_l1_ratio)

    else:
        raise ValueError("model_type must be one of: simple, ridge, lasso, elasticnet")

In [ ]:
retrain_logistic_wrapper(df, "Group", model_type="simple", model_path="/content/drive/MyDrive/RepoGo_Rasp/models/logistic_model.pkl")

{'message': 'Model retrained on full dataset', 'num_samples': 84}

## Training Data summeary

In [ ]:
def training_data_summary(df, label_col):
    """
    Returns training data summary (group-wise describe)
    → frontend-ready DataFrame
    """

    validate_input(df, label_col)

    # Select numeric features only
    feature_cols = df.select_dtypes(include=["number"]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available")

    # Subset
    df_model = df[feature_cols + [label_col]].copy()

    # Remove NaNs (optional but safer)
    df_model = df_model.dropna()

    # Group-wise describe
    summary = df_model.groupby(label_col).describe()

    # Flatten columns → frontend friendly
    summary.columns = [f"{col[0]}_{col[1]}" for col in summary.columns]

    # Reset index
    summary = summary.reset_index()

    result = summary.to_dict(orient="records")

    artifact = {
        "train_summary": result
    }

    model_path = "/content/drive/MyDrive/RepoGo_Rasp/models/train_data_summary.pkl"

    joblib.dump(artifact, model_path)

    return result

## Health Score Creation

In [ ]:
df

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
0,74,0.30,2.2,61,47,31,33,0
1,20,1.20,1.0,100,80,100,70,1
2,73,0.20,1.7,67,45,44,32,0
3,22,1.20,0.8,96,79,92,72,1
4,79,0.19,1.8,70,43,41,28,0
...,...,...,...,...,...,...,...,...
79,12,1.80,0.6,84,91,96,71,1
80,87,1.80,2.2,73,44,35,33,0
81,56,0.14,1.5,71,52,42,17,0
82,24,1.30,1.1,98,79,98,83,1


In [ ]:
def compute_health_score_EW(df, positive_indicators, negative_indicators):
    df_copy = df.copy()

    # All indicators
    all_indicators = positive_indicators + negative_indicators

    # Normalize using SAME formula everywhere
    df_norm = df[all_indicators].copy()

    # Normalize positive indicators
    for col in positive_indicators:
        min_val = df[col].min()
        max_val = df[col].max()

        if max_val == min_val:
            df_norm[col] = 0  # avoid division by zero
        else:
            df_norm[col] = (df[col] - min_val) / (max_val - min_val)

    # Normalize negative indicators (invert)
    for col in negative_indicators:
        min_val = df[col].min()
        max_val = df[col].max()

        if max_val == min_val:
            df_norm[col] = 0
        else:
            df_norm[col] = (max_val - df[col]) / (max_val - min_val)

    # Health Score (Equal weight)
    df_copy["Health_Score_EW"] = (df_norm[all_indicators].mean(axis=1))*100

    return df_copy

In [ ]:
# Indicators where higher = better
positive_indicators = ["LDL_uptake", "Vascular_Marker", "Cell_Signalling", "Tube_formation","In_vivo_recovery"]

# Indicators where higher = worse (need inversion)
negative_indicators = ["Permeability", "Total_ROS"]

# All indicators (7 total)
all_indicators = positive_indicators + negative_indicators

In [ ]:
compute_health_score_EW(df, positive_indicators, negative_indicators)

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group,Health_Score_EW
0,74,0.30,2.2,61,47,31,33,0,14.310353
1,20,1.20,1.0,100,80,100,70,1,79.944824
2,73,0.20,1.7,67,45,44,32,0,21.088761
3,22,1.20,0.8,96,79,92,72,1,77.957987
4,79,0.19,1.8,70,43,41,28,0,18.497505
...,...,...,...,...,...,...,...,...,...
79,12,1.80,0.6,84,91,96,71,1,83.987774
80,87,1.80,2.2,73,44,35,33,0,26.049801
81,56,0.14,1.5,71,52,42,17,0,24.515530
82,24,1.30,1.1,98,79,98,83,1,80.534700


In [ ]:
def train_health_score_EW(df, target_col="Health_Score_EW", label_col=None, model_path="/content/drive/MyDrive/RepoGo_Rasp/models/Health_Score_EW.pkl"):

    # select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # remove target
    if target_col not in feature_cols:
        raise ValueError("Target column not found")
    feature_cols.remove(target_col)

    # remove label (if exists)
    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    # split data
    X = df[feature_cols].copy()
    y = df[target_col].copy()

    # check missing values
    if X.isnull().sum().sum() > 0:
        raise ValueError("Missing values found")

    # scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # train model
    model = LinearRegression()
    model.fit(X_scaled, y)

    # predictions
    y_pred = model.predict(X_scaled)

    # metrics
    metrics = {
        "r2": r2_score(y, y_pred),
        "mae": mean_absolute_error(y, y_pred),
        "rmse": np.sqrt(mean_squared_error(y, y_pred))
    }

    # save artifact
    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "metrics": metrics,
        "features": feature_cols
    }

In [ ]:
Health_Score_EW = compute_health_score_EW(df, positive_indicators, negative_indicators)

In [ ]:
Health_Score_EW

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group,Health_Score_EW
0,74,0.30,2.2,61,47,31,33,0,14.310353
1,20,1.20,1.0,100,80,100,70,1,79.944824
2,73,0.20,1.7,67,45,44,32,0,21.088761
3,22,1.20,0.8,96,79,92,72,1,77.957987
4,79,0.19,1.8,70,43,41,28,0,18.497505
...,...,...,...,...,...,...,...,...,...
79,12,1.80,0.6,84,91,96,71,1,83.987774
80,87,1.80,2.2,73,44,35,33,0,26.049801
81,56,0.14,1.5,71,52,42,17,0,24.515530
82,24,1.30,1.1,98,79,98,83,1,80.534700


In [ ]:
train_health_score_EW(Health_Score_EW, target_col="Health_Score_EW", label_col="Group", model_path="/content/drive/MyDrive/RepoGo_Rasp/models/Health_Score_EW.pkl")

{'model_path': '/content/drive/MyDrive/RepoGo_Rasp/models/Health_Score_EW.pkl',
 'metrics': {'r2': 1.0,
  'mae': 7.718693409298707e-15,
  'rmse': np.float64(1.0931314867776796e-14)},
 'features': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery']}